# 00 — Build and check the input datasets

QuantumXCT ships **no data**. This notebook rebuilds both inputs from scratch and checks
them with the same loaders the analyses use, so every later notebook starts from verified data.

| Dataset | Built by | Source |
|---|---|---|
| Synthetic tissue (`simCellsSpace/sim_cells_datasets_co_mo_simulated_classical.h5ad`) | `data_prep/prepare_sim.py` | spatial LR → receptor → GRN simulator, seeded |
| Ovarian cancer–fibroblast co-culture (`dataset_co_mo2.h5ad`) | `data_prep/pull_ovarian.py` | GEO **GSE224333** counts + the cell annotations committed in `data_prep/` |

**Run it** in Jupyter (from anywhere inside `python/`), or headless with `python run_all.py --only data`.
A dataset that already exists is reused and re-checked. Environment switches:

* `QXCT_REBUILD=1` rebuilds both even if they exist.
* `QXCT_SKIP_OVARIAN=1` skips the ovarian download (synthetic only).

**Reproducibility note.** The simulator is fully seeded here (`SEED = 42`). The synthetic file
used in the paper was generated before count-level seeding, so a freshly simulated dataset
gives numbers close to, but not bit-identical with, the published ones. The ovarian object is
rebuilt from the public GEO counts plus the committed annotations.

The ovarian step needs internet access and `GEOparse` (`pip install GEOparse`).

In [ ]:
import os, sys, subprocess, time

# Run from the QuantumXCT/python root, wherever Jupyter was started.
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "qxct")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

SIM_PATH = os.path.join("simCellsSpace", "sim_cells_datasets_co_mo_simulated_classical.h5ad")
OV_PATH = "dataset_co_mo2.h5ad"
SEED = 42
REBUILD = os.environ.get("QXCT_REBUILD", "0") == "1"            # rebuild even if present
BUILD_OVARIAN = os.environ.get("QXCT_SKIP_OVARIAN", "0") != "1"  # needs internet (GEO)


def run(script_and_args):
    """Run a data_prep script with this Python and stream a short log."""
    cmd = [sys.executable, *script_and_args]
    print("$", " ".join(script_and_args), flush=True)
    t0 = time.time()
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-4000:])
    if p.returncode != 0:
        print(p.stderr[-4000:])
        raise RuntimeError(f"failed (exit {p.returncode}): {' '.join(script_and_args)}")
    print(f"[done in {time.time() - t0:.1f} s]")


status = {}
print("python root:", _root)
print("rebuild:", REBUILD, "| ovarian:", "build/check" if BUILD_OVARIAN else "skipped")

## 1. Synthetic tissue (the simulator)

Four populations on a 2-D layout: cell types A and B, each either isolated (`*_Lone`, the
mono-culture baseline) or in contact (`*_Interact`, the co-culture state). Contact switches on
the programmed network, which is the answer key in `simCellsSpace/ground_truth.py`.

In [ ]:
if REBUILD or not os.path.exists(SIM_PATH):
    run(["data_prep/prepare_sim.py", "--seed", str(SEED), "--out", SIM_PATH])
    status["synthetic"] = "built"
else:
    print(f"{SIM_PATH} exists -> reusing it (QXCT_REBUILD=1 to rebuild)")
    status["synthetic"] = "reused"

In [ ]:
import scanpy as sc
from analysis_tools.data import load_sim, SIM_CT1, SIM_CT2
from simCellsSpace.ground_truth import synthetic_ground_truth

ad_sim = sc.read_h5ad(SIM_PATH)
print(ad_sim)
expected = {"Type_A_Lone", "Type_B_Lone", "Type_A_Interact", "Type_B_Interact"}
assert set(ad_sim.obs["cell_type"].astype(str).unique()) == expected, ad_sim.obs["cell_type"].unique()
assert "counts" in ad_sim.layers, "raw-count layer 'counts' is missing"
missing = [g for g in SIM_CT1 + SIM_CT2 if g not in ad_sim.var_names]
assert not missing, f"panel genes missing: {missing}"

H_sim = load_sim(SIM_PATH)            # exactly what the analyses read
for k, h in H_sim.items():
    assert sum(h.values()) > 0, f"empty histogram: {k}"
    print(f"  {k}: {sum(h.values())} cells, {len(h)} distinct binary states")
print(ad_sim.obs["cell_type"].value_counts().to_string())
print("ground-truth edges:", synthetic_ground_truth())
print("synthetic dataset OK")

In [ ]:
import matplotlib.pyplot as plt
from qxct.plotting import data_histogram
os.makedirs("results/data", exist_ok=True)

def data_grid(H, ct1, ct2, name, labels=("mono-culture (basal)", "co-culture (target)")):
    """2x2 joint histograms (raw counts): rows = condition, columns = cell type."""
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    for r, (key, lab) in enumerate(zip(("mo", "co"), labels)):
        for c, (ct, gl) in enumerate((("ct1", ct1), ("ct2", ct2))):
            data_histogram(axes[r, c], {k: int(v) for k, v in H[f"{ct}_{key}"].items()}, len(gl),
                           f"{name} {ct.upper()} - {lab}")
    fig.tight_layout(); fig.savefig(f"results/data/{name}_data_hist_grid.png", dpi=150)
    plt.show()

data_grid(load_sim(SIM_PATH), SIM_CT1, SIM_CT2, "synthetic")

## 2. Ovarian cancer–fibroblast co-culture (GEO GSE224333)

Downloads the raw 10x counts, re-attaches the committed cell annotations
(`data_prep/dataset_co_mo2_cell_metadata.csv`, `data_prep/dataset_co_mo2_genes.txt`) per
sample, and writes `dataset_co_mo2.h5ad`. The dry run first prints which GEO sample maps to
which group (cancer / fibroblast × mono / co-culture).

In [ ]:
if not BUILD_OVARIAN:
    print("QXCT_SKIP_OVARIAN=1 -> skipping the ovarian download")
    status["ovarian"] = "skipped"
elif REBUILD or not os.path.exists(OV_PATH):
    run(["data_prep/pull_ovarian.py", "--dry-run"])          # check the sample -> group mapping
    run(["data_prep/pull_ovarian.py", "--out", OV_PATH])
    status["ovarian"] = "built"
else:
    print(f"{OV_PATH} exists -> reusing it (QXCT_REBUILD=1 to rebuild)")
    status["ovarian"] = "reused"

In [ ]:
if status["ovarian"] != "skipped":
    from analysis_tools.data import load_ovarian, OV_CT1, OV_CT2

    ad_ov = sc.read_h5ad(OV_PATH)
    print(ad_ov)
    groups = {"Cancer Cells (Mo)", "Cancer Cells (Co)", "Fibroblasts (Mo)", "Fibroblasts (Co)"}
    found = set(ad_ov.obs["BatchID"].astype(str).unique())
    assert groups <= found, f"missing groups: {groups - found}"
    missing = [g for g in OV_CT1 + OV_CT2 if g not in ad_ov.var_names]
    assert not missing, f"panel genes missing: {missing}"

    H_ov = load_ovarian(OV_PATH)       # exactly what the analyses read
    for k, h in H_ov.items():
        assert sum(h.values()) > 0, f"empty histogram: {k}"
        print(f"  {k}: {sum(h.values())} cells, {len(h)} distinct binary states")
    print(ad_ov.obs["BatchID"].value_counts().to_string())
    print("ovarian dataset OK")

In [ ]:
if status["ovarian"] != "skipped":
    data_grid(load_ovarian(OV_PATH), OV_CT1, OV_CT2, "ovarian")

## 3. Summary

Both files are ignored by git (`*.h5ad`); rerun this notebook whenever you need them.
Next: `qxct_pred_classic_sim_oo.ipynb` (synthetic) and `qxct_ovarian_cocultured_oo.ipynb` (ovarian).

In [ ]:
import pandas as pd

rows = []
for name, path in (("synthetic", SIM_PATH), ("ovarian", OV_PATH)):
    rows.append({"dataset": name, "file": path, "status": status.get(name, "-"),
                 "size_MB": round(os.path.getsize(path) / 1e6, 1) if os.path.exists(path) else None})
pd.DataFrame(rows)